# Pipeline de Dados — PDFs Institucionais

### Objetivo:
Extrair e analisar o vocabulário de relatórios institucionais do setor energético brasileiro para mapear a evolução do discurso sobre tecnologias de aquecimento de água ao longo do tempo.

### Fontes:
  - EPE: Balanço Energético Nacional (BEN)
  - PROCEL: Resultados do Programa Nacional de Conservação de Energia Elétrica
  - ANEEL: Realizações Anuais / Retrospectiva ANEEL

### Estrutura por período (1 PROCEL + 2 EPE + 1 ANEEL):
  - 2010–2015: ben2010, procel2012, aneel2014, ben2015
  - 2016–2020: ben2017, procel2018, aneel2019, ben2020
  - 2021–2024: ben2021, procel2022, aneel2023, ben2024

### Abordagem de análise textual em duas camadas:
- Camada 1 — spaCy: lematização para análise geral
- Camada 2 — busca direta: termos compostos específicos do setor de aquecimento de água

In [1]:
# ------------------------------------------------------------
# 1. MONTAGEM DO GOOGLE DRIVE
# ------------------------------------------------------------
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# ------------------------------------------------------------
# 2. IMPORTAÇÃO DAS BIBLIOTECAS
# ------------------------------------------------------------
!pip install pdfplumber spacy wordcloud -q
!python -m spacy download pt_core_news_sm -q

import os
import pdfplumber
import spacy
import matplotlib.pyplot as plt
from collections import Counter
from wordcloud import WordCloud

nlp = spacy.load("pt_core_news_sm")

# Os Balanços Energéticos (BEN) são bilíngues (PT/EN). Sem remover o inglês,
# a nuvem de palavras fica poluída com "energy", "consumption", "table" etc.
# Combinamos as stopwords de inglês do spaCy com termos-conteúdo recorrentes
# nas tabelas em inglês desses relatórios.
from spacy.lang.en.stop_words import STOP_WORDS as STOPWORDS_EN
STOPWORDS_EN = set(STOPWORDS_EN) | {
    "energy", "consumption", "balance", "coal", "plants", "plant", "supply",
    "electricity", "fuel", "power", "heat", "demand", "output", "share",
    "year", "table", "figure", "source", "other", "primary", "production",
    "residential", "sector", "wood", "biomass", "transformation", "losses",
    "imports", "exports", "total", "natural", "oil", "data", "value", "note"
}

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 85.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 86.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
# ------------------------------------------------------------
# 3. CONFIGURAÇÃO
# ------------------------------------------------------------
pasta_pdfs = "/content/drive/MyDrive/dados_projeto_ciencia_dados/pdfs_institucionais"

# Agrupamento de arquivos por período
# Chave: rótulo do período | Valor: intervalo de anos
PERIODOS = {
    "2010–2015": range(2010, 2016),
    "2016–2020": range(2016, 2021),
    "2021–2024": range(2021, 2025)
}

# Termos compostos específicos do setor de aquecimento de água
# Buscados como sequências exatas no texto — não são lematizados
TERMOS_COMPOSTOS = [
    "aquecedor solar",
    "coletor solar",
    "bomba de calor",
    "reservatório térmico",
    "chuveiro elétrico",
    "aquecedor a gás",
    "eficiência energética",
    "energia solar"
]

In [4]:
# ------------------------------------------------------------
# 4. COLETA DOS DADOS
# Extração de texto bruto dos PDFs por período
# ------------------------------------------------------------

def extrair_texto_pdf(caminho):
    """
    Extrai texto de todas as páginas de um PDF.
    Ignora páginas sem texto detectável (ex: páginas de imagem).
    """
    texto = ""
    with pdfplumber.open(caminho) as pdf:
        for pagina in pdf.pages:
            t = pagina.extract_text()
            if t:
                texto += t + " "
    return texto


# Percorrer PDFs e agrupar texto por período
textos_brutos    = {periodo: "" for periodo in PERIODOS}
arquivos_lidos   = {periodo: [] for periodo in PERIODOS}

print("Extraindo texto dos PDFs...\n")

for arquivo in sorted(os.listdir(pasta_pdfs)):
    if not arquivo.endswith(".pdf"):
        continue
    try:
        ano = int(arquivo[:4])
    except ValueError:
        print(f"  Ignorado (sem ano no nome): {arquivo}")
        continue

    caminho = os.path.join(pasta_pdfs, arquivo)

    for periodo, intervalo in PERIODOS.items():
        if ano in intervalo:
            print(f"  Lendo {arquivo} → {periodo}")
            textos_brutos[periodo] += extrair_texto_pdf(caminho)
            arquivos_lidos[periodo].append(arquivo)
            break

print("\nArquivos por período:")
for periodo, arquivos in arquivos_lidos.items():
    print(f"  {periodo}: {len(arquivos)} PDFs")
    for a in arquivos:
        print(f"    - {a}")


Extraindo texto dos PDFs...

  Lendo 2010_epe_ben.pdf → 2010–2015
  Lendo 2012_procel_resultados.pdf → 2010–2015
  Lendo 2014_aneel_relatorio.pdf → 2010–2015
  Lendo 2015_epe_ben.pdf → 2010–2015
  Lendo 2017_epe_ben.pdf → 2016–2020
  Lendo 2018_procel_resultados.pdf → 2016–2020
  Lendo 2019_aneel_relatorio.pdf → 2016–2020
  Lendo 2020_epe_ben.pdf → 2016–2020
  Lendo 2021_epe_ben.pdf → 2021–2024
  Lendo 2022_procel_resultados.pdf → 2021–2024
  Lendo 2023_aneel_relatorio.pdf → 2021–2024


  Lendo 2024_epe_ben.pdf → 2021–2024

Arquivos por período:
  2010–2015: 4 PDFs
    - 2010_epe_ben.pdf
    - 2012_procel_resultados.pdf
    - 2014_aneel_relatorio.pdf
    - 2015_epe_ben.pdf
  2016–2020: 4 PDFs
    - 2017_epe_ben.pdf
    - 2018_procel_resultados.pdf
    - 2019_aneel_relatorio.pdf
    - 2020_epe_ben.pdf
  2021–2024: 4 PDFs
    - 2021_epe_ben.pdf
    - 2022_procel_resultados.pdf
    - 2023_aneel_relatorio.pdf
    - 2024_epe_ben.pdf


In [5]:
# ------------------------------------------------------------
# 5. INSPEÇÃO INICIAL
# Verificar volume de texto extraído por período
# ------------------------------------------------------------

print("\nVolume de texto extraído por período:")
for periodo, texto in textos_brutos.items():
    print(f"  {periodo}: {len(texto):,} caracteres")



Volume de texto extraído por período:
  2010–2015: 1,634,642 caracteres
  2016–2020: 1,525,779 caracteres
  2021–2024: 1,494,303 caracteres


In [ ]:
# ------------------------------------------------------------
# 6. CAMADA 1 — PROCESSAMENTO COM spaCy
# Lematização em chunks para contornar limite de caracteres
# O spaCy tem limite padrão de 1.000.000 caracteres por chamada.
# Textos maiores são divididos em pedaços de 500.000 caracteres,
# processados separadamente e depois reunidos.
# ------------------------------------------------------------

def processar_spacy(texto, tamanho_chunk=500_000):
    """
    Processa texto longo com spaCy dividindo em chunks.
    - Divide o texto em pedaços de tamanho_chunk caracteres
    - Processa cada pedaço separadamente
    - Reúne todos os tokens ao final
    """
    tokens = []
    inicio = 0

    while inicio < len(texto):
        chunk = texto[inicio: inicio + tamanho_chunk]
        doc = nlp(chunk.lower())
        tokens_chunk = [
            token.lemma_
            for token in doc
            if not token.is_stop
            and not token.is_punct
            and not token.like_num
            and token.is_alpha
            and token.lemma_ not in STOPWORDS_EN
            and len(token.text) >= 4
        ]
        tokens.extend(tokens_chunk)
        inicio += tamanho_chunk

    return tokens


print("\nProcessando textos com spaCy (em chunks)...")
tokens_por_periodo = {}
for periodo, texto in textos_brutos.items():
    tokens = processar_spacy(texto)
    tokens_por_periodo[periodo] = tokens
    print(f"  {periodo}: {len(tokens):,} tokens")


Processando textos com spaCy (em chunks)...


In [ ]:
# ------------------------------------------------------------
# 6b. NUVENS DE PALAVRAS POR PERÍODO (Camada 1 — spaCy)
# Visualiza o vocabulário dominante de cada período, atendendo ao
# objetivo de mapear a evolução terminológica do setor energético.
# ------------------------------------------------------------

fig, axes = plt.subplots(1, len(tokens_por_periodo), figsize=(18, 6))

for ax, (periodo, tokens) in zip(axes, tokens_por_periodo.items()):
    frequencias = Counter(tokens)
    nuvem = WordCloud(
        width=600, height=600,
        background_color="white",
        colormap="viridis",
        max_words=80
    ).generate_from_frequencies(frequencias)

    ax.imshow(nuvem, interpolation="bilinear")
    ax.set_title(f"Vocabulário {periodo}", fontsize=13)
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# ------------------------------------------------------------
# 7. CAMADA 2 — BUSCA DIRETA POR TERMOS COMPOSTOS
# Contagem de ocorrências exatas no texto bruto
# Preserva o sentido dos termos de múltiplas palavras
# ------------------------------------------------------------

def contar_termos_compostos(texto, termos):
    """
    Busca e conta ocorrências exatas de termos compostos
    no texto em minúsculas, sem lematização.
    Garante que "coletor solar" seja contado como unidade,
    não como "coletor" e "solar" separadamente.
    """
    texto_lower = texto.lower()
    return {termo: texto_lower.count(termo) for termo in termos}


contagens_por_periodo = {}
for periodo, texto in textos_brutos.items():
    contagens_por_periodo[periodo] = contar_termos_compostos(
        texto, TERMOS_COMPOSTOS
    )




In [ ]:
# ------------------------------------------------------------
# 8. LIMPEZA E TRATAMENTO
# Organizar contagens em DataFrame para análise e exportação
# ------------------------------------------------------------
import pandas as pd

linhas = []
for periodo, contagens in contagens_por_periodo.items():
    for termo, freq in contagens.items():
        linhas.append({
            "periodo"   : periodo,
            "termo"     : termo,
            "frequencia": freq
        })

df_termos = pd.DataFrame(linhas)

# Tabela pivotada para visualização comparativa
df_pivot = df_termos.pivot(
    index="termo",
    columns="periodo",
    values="frequencia"
).fillna(0).astype(int)

print("\nFrequência dos termos compostos por período:\n")
print(df_pivot.to_string())

In [ ]:
# ------------------------------------------------------------
# 9. VERIFICAÇÃO PÓS-LIMPEZA
# Períodos com zero ocorrências em todos os termos
# ------------------------------------------------------------

for periodo in PERIODOS:
    total = df_termos[df_termos["periodo"] == periodo]["frequencia"].sum()
    print(f"  {periodo}: {total} ocorrências totais")


In [ ]:
# ------------------------------------------------------------
# 10. VISUALIZAÇÕES
# Como evoluiu a frequência dos termos compostos ao longo
# dos períodos nos documentos institucionais?
# ------------------------------------------------------------

periodos_ordem = list(PERIODOS.keys())
cores = [
    "steelblue", "darkorange", "teal", "purple",
    "firebrick", "olive", "slategray", "darkgreen"
]

fig, ax = plt.subplots(figsize=(12, 6))

for i, termo in enumerate(TERMOS_COMPOSTOS):
    frequencias = [
        contagens_por_periodo[p].get(termo, 0)
        for p in periodos_ordem
    ]
    ax.plot(
        periodos_ordem,
        frequencias,
        marker="o",
        label=termo,
        color=cores[i]
    )

ax.set_title(
    "Evolução da Frequência de Termos Compostos\n"
    "em Relatórios Institucionais do Setor Energético — 2010 a 2024",
    fontsize=13
)
ax.set_xlabel("Período")
ax.set_ylabel("Número de ocorrências")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


# Comparativo em barras agrupadas por período
df_pivot.T.plot(
    kind="bar",
    figsize=(14, 6),
    colormap="tab10"
)
plt.title(
    "Frequência dos Termos Compostos por Período\n"
    "Relatórios PROCEL, EPE e ANEEL — 2010 a 2024",
    fontsize=13
)
plt.xlabel("Período")
plt.ylabel("Número de ocorrências")
plt.xticks(rotation=0)
plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    title="Termo"
)
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# 11. EXPORTAÇÃO
# ------------------------------------------------------------

df_termos.to_csv(
    "/content/drive/MyDrive/dados_projeto_ciencia_dados/pdfs_frequencia_termos.csv",
    index=False,
    encoding="utf-8-sig"
)

df_pivot.to_csv(
    "/content/drive/MyDrive/dados_projeto_ciencia_dados/pdfs_frequencia_pivot.csv",
    encoding="utf-8-sig"
)

print("Arquivos exportados com sucesso.")